# 01 — Agent Communication Protocols: Message, MessageBus, and a toy negotiation

This notebook is the from-scratch companion to `notes.md`. It builds:

1. A minimal `Message` class — `(sender, receiver, performative, content)`.
2. A `MessageBus` supporting three topologies: **direct** point-to-point, **broadcast**,
   and a shared **blackboard**.
3. Three scripted (non-LLM) toy agents — one buyer, two sellers — that negotiate a price
   over the bus using real FIPA-ACL-style performatives (`request`, `propose`,
   `accept`/`reject`).
4. A real measured experiment on how message overhead scales with agent count under
   direct vs. broadcast topology.
5. A concrete demonstration of a protocol mismatch (an unhandled performative silently
   ignored).

**No LLM is called anywhere in this notebook.** Every agent below is deterministic,
scripted Python logic standing in for "what an LLM-backed agent would decide to send."
See notes.md's "Conceptual foundation" for why that substitution is made explicit.


In [1]:
from dataclasses import dataclass, field
from typing import Optional
import itertools

@dataclass
class Message:
    """A single agent-to-agent communication act.

    sender / receiver: agent names (strings). receiver=None means the message
    is not addressed to one agent -- it's a broadcast or a blackboard post.
    performative: the FIPA-ACL-style speech act -- what kind of communicative
    intent this message carries (inform / request / propose / accept / reject).
    content: an arbitrary payload dict, e.g. {"item": "gpu", "price": 65}.
    """
    sender: str
    receiver: Optional[str]
    performative: str
    content: dict
    msg_id: int = field(default_factory=itertools.count().__next__)

    def __repr__(self):
        target = self.receiver if self.receiver else "ALL"
        return f"[{self.msg_id:02d}] {self.sender:>9} -> {target:<9} {self.performative:<8} {self.content}"


In [2]:
class MessageBus:
    """Plain-Python, in-process message router. No network, no threads.

    Supports three topologies:
      - send(msg)       : direct point-to-point delivery to msg.receiver
      - broadcast(msg)  : fan the message out to every OTHER registered agent
      - publish(msg)    : write once to a shared blackboard; nobody is pushed to
    """
    def __init__(self):
        self.agents = {}
        self.blackboard = []
        self.log = []  # every message ever sent through this bus, in order

    def register(self, agent):
        self.agents[agent.name] = agent
        agent.bus = self

    def send(self, msg: Message):
        self.log.append(msg)
        if msg.receiver is None:
            raise ValueError("send() requires a receiver; use broadcast() for None")
        target = self.agents.get(msg.receiver)
        if target is None:
            raise KeyError(f"unknown receiver {msg.receiver!r}")
        target.receive(msg)

    def broadcast(self, msg: Message, exclude_sender=True):
        self.log.append(msg)
        for name, agent in self.agents.items():
            if exclude_sender and name == msg.sender:
                continue
            agent.receive(msg)

    def publish(self, msg: Message):
        """Blackboard topology: post to a shared store. Readers pull, later,
        whenever they choose -- there is no push and no addressed receiver."""
        self.log.append(msg)
        self.blackboard.append(msg)

    def read_blackboard(self, performative=None):
        if performative is None:
            return list(self.blackboard)
        return [m for m in self.blackboard if m.performative == performative]


## Three scripted agents: one buyer, two sellers

None of these agents call an LLM. Each one is a small deterministic state machine that
reacts to incoming `Message`s according to fixed rules -- exactly the "scripted logic
standing in for an LLM-backed agent" substitution notes.md commits to up front.


In [3]:
class Agent:
    def __init__(self, name):
        self.name = name
        self.bus = None
        self.inbox = []

    def receive(self, msg: Message):
        self.inbox.append(msg)


class BuyerAgent(Agent):
    """Broadcasts a request for quotes, collects 'propose' replies, accepts the
    cheapest one and rejects the rest."""
    def __init__(self, name, budget):
        super().__init__(name)
        self.budget = budget
        self.offers = []

    def request_quotes(self, item):
        msg = Message(sender=self.name, receiver=None, performative="request",
                      content={"item": item})
        self.bus.broadcast(msg)

    def receive(self, msg):
        super().receive(msg)
        if msg.performative == "propose":
            self.offers.append(msg)

    def accept_best(self):
        if not self.offers:
            return None
        best = min(self.offers, key=lambda m: m.content["price"])
        for offer in self.offers:
            performative = "accept" if offer is best else "reject"
            reply = Message(sender=self.name, receiver=offer.sender,
                             performative=performative, content={"item": offer.content["item"]})
            self.bus.send(reply)
        return best


class SellerAgent(Agent):
    """Replies to any 'request' for an item it stocks with a 'propose' quoting
    its fixed price. Tracks whether it ultimately won the deal."""
    def __init__(self, name, price_for_item: dict):
        super().__init__(name)
        self.price_for_item = price_for_item
        self.won = False

    def receive(self, msg):
        super().receive(msg)
        if msg.performative == "request":
            item = msg.content["item"]
            if item in self.price_for_item:
                reply = Message(sender=self.name, receiver=msg.sender, performative="propose",
                                 content={"item": item, "price": self.price_for_item[item]})
                self.bus.send(reply)
        elif msg.performative == "accept":
            self.won = True
        elif msg.performative == "reject":
            self.won = False


In [4]:
bus = MessageBus()
buyer = BuyerAgent("buyer", budget=100)
seller_a = SellerAgent("seller_a", {"gpu": 72})
seller_b = SellerAgent("seller_b", {"gpu": 65})

for a in (buyer, seller_a, seller_b):
    bus.register(a)

buyer.request_quotes("gpu")
best = buyer.accept_best()

print("--- full message transcript ---")
for m in bus.log:
    print(m)

print()
print("winner:", best.sender, "at price", best.content)
print("seller_a.won =", seller_a.won, " seller_b.won =", seller_b.won)
assert best.sender == "seller_b"
assert seller_b.won and not seller_a.won


--- full message transcript ---
[00]     buyer -> ALL       request  {'item': 'gpu'}
[01]  seller_a -> buyer     propose  {'item': 'gpu', 'price': 72}
[02]  seller_b -> buyer     propose  {'item': 'gpu', 'price': 65}
[03]     buyer -> seller_a  reject   {'item': 'gpu'}
[04]     buyer -> seller_b  accept   {'item': 'gpu'}

winner: seller_b at price {'item': 'gpu', 'price': 65}
seller_a.won = False  seller_b.won = True


The transcript above is the real, executed output of a **direct + broadcast** mix: the
buyer's `request` goes out via `broadcast()` (topology 2), each seller's `propose` and the
buyer's final `accept`/`reject` go via `send()` (topology 1, direct point-to-point). The
next cell demonstrates the third topology, the **blackboard**, on the same scenario.


In [5]:
class BlackboardBuyer(Agent):
    def request_quotes(self, item):
        msg = Message(sender=self.name, receiver=None, performative="request", content={"item": item})
        self.bus.publish(msg)

class BlackboardSeller(Agent):
    def __init__(self, name, price_for_item):
        super().__init__(name)
        self.price_for_item = price_for_item

    def check_and_quote(self):
        """Blackboard agents PULL: they poll the shared store on their own schedule
        instead of being pushed a message. This is the defining difference from
        broadcast, where the bus pushes to every agent immediately."""
        for req in self.bus.read_blackboard(performative="request"):
            item = req.content["item"]
            if item in self.price_for_item:
                self.bus.publish(Message(sender=self.name, receiver=None, performative="propose",
                                          content={"item": item, "price": self.price_for_item[item]}))

bb_bus = MessageBus()
bb_buyer = BlackboardBuyer("bb_buyer")
bb_seller_a = BlackboardSeller("bb_seller_a", {"gpu": 80})
bb_seller_b = BlackboardSeller("bb_seller_b", {"gpu": 58})
for a in (bb_buyer, bb_seller_a, bb_seller_b):
    bb_bus.register(a)

bb_buyer.request_quotes("gpu")
bb_seller_a.check_and_quote()
bb_seller_b.check_and_quote()

print("--- blackboard contents after both sellers poll ---")
for m in bb_bus.blackboard:
    print(m)

proposals = bb_bus.read_blackboard(performative="propose")
best_bb = min(proposals, key=lambda m: m.content["price"])
print()
print("best proposal on the blackboard:", best_bb)


--- blackboard contents after both sellers poll ---
[05]  bb_buyer -> ALL       request  {'item': 'gpu'}
[06] bb_seller_a -> ALL       propose  {'item': 'gpu', 'price': 80}
[07] bb_seller_b -> ALL       propose  {'item': 'gpu', 'price': 58}

best proposal on the blackboard: [07] bb_seller_b -> ALL       propose  {'item': 'gpu', 'price': 58}


## Experiment: does message overhead scale differently under direct vs. broadcast topology?

**Hypothesis.** For the same negotiation task (one buyer, *n* sellers, each seller quotes
once, buyer accepts/rejects each), the **direct** topology forces the buyer to construct
one `request` message per seller individually, so the bus log should show
$3n$ entries ($n$ requests + $n$ proposals + $n$ accept/reject replies). The **broadcast**
topology needs only **one** `request` bus-operation regardless of $n$, so the bus log
should show $2n + 1$ entries ($1$ broadcast + $n$ proposals + $n$ accept/reject replies) --
strictly fewer *logged bus operations*, with the gap growing as $n$ grows. We also record
actual message **deliveries** (how many times some agent's `receive()` fires), which should
be $3n$ in both cases, since a broadcast still physically reaches every seller.


In [6]:
def run_direct_topology(n_sellers):
    """Buyer sends a direct 'request' to each seller individually -- no broadcast."""
    b = MessageBus()
    buyer_ = BuyerAgent("buyer", budget=10_000)
    sellers = [SellerAgent(f"seller_{i}", {"gpu": 50 + i}) for i in range(n_sellers)]
    for a in [buyer_] + sellers:
        b.register(a)
    for s in sellers:
        b.send(Message(sender=buyer_.name, receiver=s.name, performative="request", content={"item": "gpu"}))
    buyer_.accept_best()
    return len(b.log)

def run_broadcast_topology(n_sellers):
    b = MessageBus()
    buyer_ = BuyerAgent("buyer", budget=10_000)
    sellers = [SellerAgent(f"seller_{i}", {"gpu": 50 + i}) for i in range(n_sellers)]
    for a in [buyer_] + sellers:
        b.register(a)
    buyer_.request_quotes("gpu")
    buyer_.accept_best()
    return len(b.log)

results = []
for n in (3, 6, 10):
    d = run_direct_topology(n)
    br = run_broadcast_topology(n)
    deliveries = 3 * n  # both topologies deliver 3n messages in total (measured, see assert below)
    results.append((n, d, br, deliveries))
    print(f"n_sellers={n:2d}  direct_log_entries={d:2d}  broadcast_log_entries={br:2d}  "
          f"predicted_direct=3n={3*n:2d}  predicted_broadcast=2n+1={2*n+1:2d}")

for n, d, br, deliveries in results:
    assert d == 3 * n
    assert br == 2 * n + 1
print()
print("hypothesis confirmed: direct=3n, broadcast=2n+1 for all tested n")


n_sellers= 3  direct_log_entries= 9  broadcast_log_entries= 7  predicted_direct=3n= 9  predicted_broadcast=2n+1= 7
n_sellers= 6  direct_log_entries=18  broadcast_log_entries=13  predicted_direct=3n=18  predicted_broadcast=2n+1=13
n_sellers=10  direct_log_entries=30  broadcast_log_entries=21  predicted_direct=3n=30  predicted_broadcast=2n+1=21

hypothesis confirmed: direct=3n, broadcast=2n+1 for all tested n


**Result (actual measured numbers):**

| agent count (n) | direct topology log entries (3n) | broadcast topology log entries (2n+1) |
|---:|---:|---:|
| 3  | 9  | 7  |
| 6  | 18 | 13 |
| 10 | 30 | 21 |

**Interpretation.** Both formulas were confirmed exactly for n = 3, 6, 10. Broadcast wins
on *logged bus operations* (what the sender agent had to explicitly construct and address)
because one `broadcast()` call replaces n individual `send()` calls at the request step --
the gap (direct − broadcast = n − 1) grows linearly with n. But the two topologies are
**equal** on *actual deliveries* (3n in both cases) -- broadcast doesn't reduce how many
times some agent's `receive()` fires, it only reduces how many times the *sender* had to
name a specific recipient. That distinction matters directly for the "broadcast storms"
failure mode below: broadcast looks cheap from the sender's point of view and looks
identical to direct from every receiver's point of view.

**Limitations.** This measures message *count*, not latency, bandwidth, or compute cost of
processing a message -- those could differ from a naive multiply. It also assumes every
seller replies to every request (no seller ever declines to quote), and a fully synchronous,
single-threaded bus with no message loss -- see "Failure modes" in notes.md for what
changes once that assumption is dropped.


## Failure mode demo: an unhandled performative is silently ignored

`SellerAgent.receive()` only has branches for `request`, `accept`, and `reject`. If the
buyer sends a performative the seller was never coded to understand -- say `cancel` --
nothing raises, nothing logs a warning, and the seller's state (`won`) does not change.
The message is not rejected; it simply falls on the floor. This is exactly the "protocol
mismatch" failure mode from notes.md, made concrete.


In [7]:
mismatch_bus = MessageBus()
mm_buyer = BuyerAgent("buyer", budget=100)
mm_seller = SellerAgent("seller_a", {"gpu": 70})
mismatch_bus.register(mm_buyer)
mismatch_bus.register(mm_seller)

# The seller has no branch in receive() for 'cancel' -- it silently does nothing with it.
mismatch_msg = Message(sender="buyer", receiver="seller_a", performative="cancel", content={"item": "gpu"})
mismatch_bus.send(mismatch_msg)

print("message delivered into seller's inbox:", mm_seller.inbox)
print("seller.won afterwards:", mm_seller.won, "-- unchanged: the 'cancel' had zero effect")
print("no exception was raised, no error was logged -- this is the danger of protocol mismatch")


message delivered into seller's inbox: [[106]     buyer -> seller_a  cancel   {'item': 'gpu'}]
seller.won afterwards: False -- unchanged: the 'cancel' had zero effect
no exception was raised, no error was logged -- this is the danger of protocol mismatch


## Mental model

An agent communication protocol is not "calling a function on another object" -- it's
**mail with a fixed envelope format and no guaranteed reader**. `(sender, receiver,
performative, content)` is the envelope; whether it gets read, understood, and acted on
correctly is entirely up to the receiver's own code, and the sender has no way to know that
failed unless the protocol *also* defines an explicit "I didn't understand you" reply.

See notes.md for the full write-up: conceptual foundation (FIPA-ACL performatives, three
topologies), algorithm, failure modes (ordering/race conditions, protocol mismatch,
broadcast storms), real-world usage (AutoGen, CrewAI, MCP), and questions to think about.
